# ACT Basics

A fundamentals notebook covering Steps 1–8. Use a **Colab GPU runtime** and run the steps in order. Step 7 trains for 20 epochs; Step 8 reloads the best checkpoint and runs a short simulator rollout.


## Setup

Run this cell once, then **restart the Colab session** before starting Step 1. After restarting, continue at Step 1.


In [ ]:
%pip install -q "lerobot[dataset]==0.6.1" "gym-aloha==0.1.5" "imageio[ffmpeg]"


## Step 1: Load demonstrations and split episodes


In [ ]:
# Step 1: Load demonstrations and split whole episodes
import random

import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset

repo_id = 'lerobot/aloha_sim_transfer_cube_human'
train_ratio = 0.8
seed = 42

dataset = LeRobotDataset(repo_id, video_backend='pyav')

episode_ids = list(range(dataset.num_episodes))
random.Random(seed).shuffle(episode_ids)

split = int(train_ratio * len(episode_ids))
train_episodes = sorted(episode_ids[:split])
val_episodes = sorted(episode_ids[split:])


## Step 2: Create action chunks


In [ ]:
# Step 2: Create action chunks and their padding masks
chunk_size = 100
delta_timestamps = {'action': [i / dataset.fps for i in range(chunk_size)]}

train_dataset = LeRobotDataset(
    repo_id,
    episodes=train_episodes,
    delta_timestamps=delta_timestamps,
    video_backend='pyav',
)

val_dataset = LeRobotDataset(
    repo_id,
    episodes=val_episodes,
    delta_timestamps=delta_timestamps,
    video_backend='pyav',
)


## Step 3: Prepare training batches


In [ ]:
# Step 3.1: Imports and batch settings
from torch.utils.data import DataLoader, default_collate
from torchvision.transforms import v2

image_key = 'observation.images.top'
batch_size = 8
image_size = (224, 224)

# Step 3.2: Calculate statistics from real training frames
raw_train = train_dataset.hf_dataset.with_format(
    'torch',
    columns=['observation.state', 'action'],
)[:]

train_stats = {}
for key, values in raw_train.items():
    train_stats[key] = {
        'mean': values.mean(dim=0),
        'std': values.std(dim=0, unbiased=False).clamp_min(1e-6),
    }


# Step 3.3: Prepare images for the pretrained ResNet-18
# LeRobot supplies float images in [0, 1].
image_transform = v2.Compose([
    v2.Resize(image_size, antialias=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


# Step 3.4: Stack samples and normalize each batch
def prepare_batch(samples):
    batch = default_collate(samples)
    batch[image_key] = image_transform(batch[image_key])

    for key, stats in train_stats.items():
        batch[key] = (batch[key] - stats['mean']) / stats['std']

    return batch  # Keep action_is_pad unchanged.


# Step 3.5: Create training and validation loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=prepare_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=prepare_batch,
)


## Step 4: Set up ResNet-18 and the state projection


In [ ]:
# Step 4.1: Imports and feature dimensions
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.ops import FrozenBatchNorm2d

state_dim = dataset.features["observation.state"]["shape"][0]
hidden_dim = 512


class ObservationEncoder(nn.Module):
    def __init__(self, state_dim, hidden_dim, weights=ResNet18_Weights.DEFAULT):
        super().__init__()

        # Step 4.2: Load ResNet-18 and keep its spatial features
        resnet = resnet18(
            weights=weights,
            norm_layer=FrozenBatchNorm2d,
        )
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])

        # Step 4.3: Project image features and robot state
        self.image_projection = nn.Conv2d(512, hidden_dim, kernel_size=1)
        self.state_projection = nn.Linear(state_dim, hidden_dim)

    # Step 4.4: Turn an observation into transformer inputs
    def forward(self, images, states):
        image_features = self.backbone(images)
        image_features = self.image_projection(image_features)
        image_tokens = image_features.flatten(2).transpose(1, 2)

        state_token = self.state_projection(states)
        state_token = state_token.unsqueeze(1)

        return image_tokens, state_token


# Step 4.5: Create the observation encoder
observation_encoder = ObservationEncoder(state_dim, hidden_dim)


## Step 5: Build the action-predicting transformer


In [ ]:
# Step 5.1: Configure the action-predicting transformer
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import (
    ACTDecoder,
    ACTEncoder,
    ACTSinusoidalPositionEmbedding2d,
)

action_dim = dataset.features["action"]["shape"][0]
act_config = ACTConfig(
    chunk_size=chunk_size, n_action_steps=chunk_size, dim_model=hidden_dim
)


class ACTActionPredictor(nn.Module):
    def __init__(self, observation_encoder, config, action_dim, feature_grid):
        super().__init__()
        self.observation_encoder = observation_encoder
        self.config = config
        self.feature_grid = feature_grid

        # Step 5.2: Create positions, transformer blocks, queries, and action head
        self.state_position = nn.Embedding(1, config.dim_model)
        self.image_positions = ACTSinusoidalPositionEmbedding2d(config.dim_model // 2)
        self.encoder = ACTEncoder(config)
        self.decoder = ACTDecoder(config)
        self.action_queries = nn.Embedding(config.chunk_size, config.dim_model)
        self.action_head = nn.Linear(config.dim_model, action_dim)

    def forward(self, images, states):
        image_tokens, state_token = self.observation_encoder(images, states)

        # Step 5.3: Encode the state and image tokens with their positions
        source = torch.cat([state_token, image_tokens], dim=1).transpose(0, 1)

        image_grid = image_tokens.transpose(1, 2).unflatten(2, self.feature_grid)
        image_pos = self.image_positions(image_grid).flatten(2).permute(2, 0, 1)
        source_pos = torch.cat([self.state_position.weight.unsqueeze(1), image_pos])

        memory = self.encoder(source, pos_embed=source_pos)

        # Step 5.4: Decode one feature vector per action position
        decoder_input = source.new_zeros(
            self.config.chunk_size, states.shape[0], self.config.dim_model
        )
        decoded = self.decoder(
            decoder_input,
            memory,
            decoder_pos_embed=self.action_queries.weight.unsqueeze(1),
            encoder_pos_embed=source_pos,
        )

        # Step 5.5: Convert each feature vector into normalized action values
        return self.action_head(decoded.transpose(0, 1))


## Step 6: Add the CVAE encoder and training objective


In [ ]:
# Step 6.1: Add the training-only CVAE encoder
from lerobot.policies.act.modeling_act import create_sinusoidal_pos_embedding


class ACTModel(ACTActionPredictor):
    def __init__(self, observation_encoder, config, state_dim, action_dim, feature_grid):
        super().__init__(observation_encoder, config, action_dim, feature_grid)

        self.vae_encoder = ACTEncoder(config, is_vae_encoder=True)
        self.summary_token = nn.Embedding(1, config.dim_model)
        self.vae_state_projection = nn.Linear(state_dim, config.dim_model)
        self.vae_action_projection = nn.Linear(action_dim, config.dim_model)
        self.latent_parameters = nn.Linear(config.dim_model, 2 * config.latent_dim)
        self.register_buffer(
            "vae_positions",
            create_sinusoidal_pos_embedding(config.chunk_size + 2, config.dim_model)
            .unsqueeze(1),
        )
        self.latent_projection = nn.Linear(config.latent_dim, config.dim_model)
        self.latent_position = nn.Embedding(1, config.dim_model)

    # Step 6.2: Encode expert actions and sample z during training
    def encode_latent(self, states, actions, padding_mask):
        summary = self.summary_token.weight.unsqueeze(0).expand(states.shape[0], -1, -1)
        state = self.vae_state_projection(states).unsqueeze(1)
        action_tokens = self.vae_action_projection(actions)
        tokens = torch.cat([summary, state, action_tokens], dim=1).transpose(0, 1)

        prefix_mask = padding_mask.new_zeros(states.shape[0], 2)
        mask = torch.cat([prefix_mask, padding_mask], dim=1)
        encoded = self.vae_encoder(
            tokens, pos_embed=self.vae_positions, key_padding_mask=mask
        )
        mu, log_variance = self.latent_parameters(encoded[0]).chunk(2, dim=-1)
        z = mu + (0.5 * log_variance).exp() * torch.randn_like(mu)
        return z, mu, log_variance

    # Step 6.3: Condition the Step 5 transformer on z
    def forward(self, images, states, actions=None, padding_mask=None):
        if self.training:
            z, mu, log_variance = self.encode_latent(states, actions, padding_mask)
        else:
            z = states.new_zeros(states.shape[0], self.config.latent_dim)
            mu = log_variance = None

        image_tokens, state_token = self.observation_encoder(images, states)
        latent_token = self.latent_projection(z).unsqueeze(1)
        source = torch.cat([latent_token, state_token, image_tokens], dim=1).transpose(0, 1)

        image_grid = image_tokens.transpose(1, 2).unflatten(2, self.feature_grid)
        image_pos = self.image_positions(image_grid).flatten(2).permute(2, 0, 1)
        source_pos = torch.cat([
            self.latent_position.weight.unsqueeze(1),
            self.state_position.weight.unsqueeze(1),
            image_pos,
        ])
        memory = self.encoder(source, pos_embed=source_pos)
        decoder_input = source.new_zeros(
            self.config.chunk_size, states.shape[0], self.config.dim_model
        )
        decoded = self.decoder(
            decoder_input,
            memory,
            decoder_pos_embed=self.action_queries.weight.unsqueeze(1),
            encoder_pos_embed=source_pos,
        )
        predictions = self.action_head(decoded.transpose(0, 1))
        return predictions, mu, log_variance


# Step 6.4: Combine masked action error and KL loss
def act_loss(predictions, targets, padding_mask, mu, log_variance, kl_weight):
    valid = (~padding_mask).unsqueeze(-1)
    errors = (predictions - targets).abs()
    valid_values = (valid.sum() * targets.shape[-1]).clamp_min(1)
    action_loss = (errors * valid).sum() / valid_values
    kl_loss = 0.5 * (
        mu.square() + log_variance.exp() - 1 - log_variance
    ).sum(dim=-1).mean()
    total_loss = action_loss + kl_weight * kl_loss
    return total_loss, action_loss, kl_loss


# Step 6.5: Create the complete model using the existing configuration
model = ACTModel(observation_encoder, act_config, state_dim, action_dim, (7, 7))


## Step 7: Train, validate, and save


In [ ]:
# Step 7.1: Set up training and the checkpoint folder
from pathlib import Path
from google.colab import drive
from tqdm.auto import tqdm

drive.mount("/content/drive")
checkpoint_dir = Path("/content/drive/MyDrive/act-basics/checkpoints/best")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

num_epochs = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training device: {device}", flush=True)
model = model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=act_config.optimizer_lr,
    weight_decay=act_config.optimizer_weight_decay,
)
batch_keys = (image_key, "observation.state", "action", "action_is_pad")


# Step 7.2: Measure masked action error using z = 0
@torch.inference_mode()
def validate():
    model.eval()
    error_sum = 0.0
    valid_values = 0

    for batch in tqdm(val_loader, desc="Validation", leave=False):
        batch = {key: batch[key].to(device) for key in batch_keys}
        predictions, _, _ = model(batch[image_key], batch["observation.state"])

        valid = (~batch["action_is_pad"]).unsqueeze(-1)
        errors = (predictions - batch["action"]).abs()
        error_sum += (errors * valid).sum().item()
        valid_values += valid.sum().item() * batch["action"].shape[-1]

    return error_sum / valid_values


# Step 7.3: Train with the Step 6 action and KL losses
best_val_l1 = float("inf")

for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss_sum = 0.0

    for batch in tqdm(
        train_loader, desc=f"Training {epoch}/{num_epochs}", leave=False
    ):
        batch = {key: batch[key].to(device) for key in batch_keys}
        optimizer.zero_grad()

        predictions, mu, log_variance = model(
            batch[image_key], batch["observation.state"],
            batch["action"], batch["action_is_pad"],
        )
        loss, _, _ = act_loss(
            predictions, batch["action"], batch["action_is_pad"],
            mu, log_variance, act_config.kl_weight,
        )
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * batch["action"].shape[0]

    train_loss = train_loss_sum / len(train_loader.dataset)
    val_l1 = validate()

    # Step 7.4: Save the model with the lowest validation error
    if val_l1 < best_val_l1:
        best_val_l1 = val_l1
        act_config.save_pretrained(checkpoint_dir)
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "state_dim": state_dim,
                "action_dim": action_dim,
                "feature_grid": model.feature_grid,
                "train_stats": train_stats,
                "image_key": image_key,
                "image_size": image_size,
                "image_mean": image_transform.transforms[1].mean,
                "image_std": image_transform.transforms[1].std,
                "repo_id": repo_id,
                "fps": dataset.fps,
                "train_episodes": train_episodes,
                "val_episodes": val_episodes,
                "epoch": epoch,
                "val_l1": val_l1,
            },
            checkpoint_dir / "model.pt",
        )

    print(
        f"Epoch {epoch}/{num_epochs} | "
        f"Train loss: {train_loss:.4f} | Val L1: {val_l1:.4f}"
    )

print(f"Best validation L1: {best_val_l1:.4f} | Saved to: {checkpoint_dir}")


## Step 8: Reload and use ACT


In [ ]:
# Step 8.1: Reload the selected model for a simulator rollout
import os

# Set these before importing the simulator in Colab.
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import gymnasium as gym
import gym_aloha  # Registers the ALOHA environments.
import imageio.v2 as imageio
from IPython.display import Video, display

project_dir = Path("/content/drive/MyDrive/act-basics")
checkpoint_dir = project_dir / "checkpoints" / "best"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    checkpoint_dir / "model.pt", map_location="cpu", weights_only=False
)
act_config = ACTConfig.from_pretrained(checkpoint_dir)
observation_encoder = ObservationEncoder(
    checkpoint["state_dim"], act_config.dim_model, weights=None
)
model = ACTModel(
    observation_encoder, act_config,
    checkpoint["state_dim"], checkpoint["action_dim"], checkpoint["feature_grid"],
)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device).eval()  # Step 6 now uses z = 0.


# Step 8.2: Restore preprocessing and prepare one simulator observation
image_transform = v2.Compose([
    v2.Resize(checkpoint["image_size"], antialias=True),
    v2.Normalize(mean=checkpoint["image_mean"], std=checkpoint["image_std"]),
])
state_stats = checkpoint["train_stats"]["observation.state"]
action_stats = checkpoint["train_stats"]["action"]


def prepare_observation(obs):
    image = torch.from_numpy(obs["pixels"]["top"]).permute(2, 0, 1).float() / 255.0
    image = image_transform(image).unsqueeze(0).to(device)

    state = torch.as_tensor(obs["agent_pos"], dtype=torch.float32)
    state = (state - state_stats["mean"]) / state_stats["std"]
    state = state.unsqueeze(0).to(device)
    return image, state

# Step 8.3: Predict a chunk and restore the original action units
@torch.inference_mode()
def predict_chunk(obs):
    image, state = prepare_observation(obs)
    normalized_actions, _, _ = model(image, state)
    actions = normalized_actions[0].cpu() * action_stats["std"] + action_stats["mean"]
    return actions.numpy()  # [chunk_size, action_dim], normally [100, 14].


# Step 8.4: Execute a few actions, then replan from the latest observation
assert checkpoint["repo_id"] == "lerobot/aloha_sim_transfer_cube_human"
execute_steps = 10
max_steps = 400
assert 1 <= execute_steps <= act_config.chunk_size

env = gym.make(
    "gym_aloha/AlohaTransferCube-v0",
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
    observation_width=640,
    observation_height=480,
    max_episode_steps=max_steps,
)
video_path = project_dir / "step8_rollout.mp4"

try:
    obs, _ = env.reset(seed=0)
    with imageio.get_writer(str(video_path), fps=checkpoint["fps"]) as video:
        video.append_data(obs["pixels"]["top"])
        for step in range(max_steps):
            if step % execute_steps == 0:
                action_chunk = predict_chunk(obs)
                if step == 0:
                    print("Action chunk:", action_chunk.shape)

            action = action_chunk[step % execute_steps]
            obs, reward, terminated, truncated, info = env.step(action)
            video.append_data(obs["pixels"]["top"])
            if terminated or truncated:
                break
finally:
    env.close()


# Step 8.5: Watch the rollout
print(f"Ran {step + 1} simulator steps.")
display(Video(str(video_path), embed=True))
